# Correlation and What It Does Not Tell You

Correlation is the most quoted dependence number in finance and the most over-read. This
notebook takes it apart: what the single number $\rho$ captures, and five things it is silent on.

## 0. Math link

This notebook applies the correlation coefficient from the probability course:

- [Correlation coefficient](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/09_correlation_coefficient.ipynb)
  (definition, bounds, invariance to units, Anscombe's quartet)
- built on [Covariance](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/06_covariance.ipynb)
  and the previous notebook, [01 Variance of a sum and a difference](./01_variance_of_sum_and_difference.ipynb).

## 1. Introduction

In 01 the correlation decided everything: diversification, tracking error, hedges. So it is
tempting to treat $\rho$ as *the* description of how two assets relate. It is not. Correlation
measures one thing, the strength of the **linear** co-movement of **standardized** returns, and
nothing else.

What correlation does **not** tell you:

1. **How much** one asset moves with the other (that is beta, not correlation).
2. **Whether they end up in the same place** (correlation ignores means; price paths can diverge).
3. **Non-linear dependence** (an option position can be fully driven by the market with $\rho \approx 0$).
4. **When** they co-move (calm-market and crash correlations differ; one number averages them).
5. **How sure you are** (sample correlations from a few years of data are very noisy).

Plus a warning about inputs: correlate **returns**, never price levels.

After this notebook you can say, for any quoted correlation, which of these questions it
answers and which it leaves open.

## 2. Theory

### Definition and bounds

$$
\rho_{XY} = \frac{\operatorname{Cov}(X,Y)}{\sigma_X\,\sigma_Y}
= \operatorname{Cov}\!\left(\frac{X-\mu_X}{\sigma_X},\ \frac{Y-\mu_Y}{\sigma_Y}\right),
\qquad -1 \le \rho \le 1
$$

The bound is the Cauchy–Schwarz inequality, $|\operatorname{Cov}(X,Y)| \le \sigma_X\sigma_Y$.
$|\rho| = 1$ exactly when $Y = a + bX$.

### What standardizing throws away

Because $\rho$ is a covariance of **standardized** variables, it is unchanged by
$X \mapsto a + bX$ with $b > 0$ (and only flips sign for $b < 0$). Two pieces of information
are discarded on purpose:

- **Location** $\mu_X, \mu_Y$: adding any drift to a return series does not move $\rho$.
- **Scale** $\sigma_X, \sigma_Y$: doubling a position does not move $\rho$.

### Correlation vs beta

The regression slope of $Y$ on $X$ puts the scale back:

$$
\beta_{Y|X} = \frac{\operatorname{Cov}(X,Y)}{\sigma_X^2} = \rho\,\frac{\sigma_Y}{\sigma_X}
$$

Same $\rho$, different $\beta$ whenever the volatility ratio differs. (Module 07/01 derives this
as a least-squares slope.)

### Uncorrelated is not independent

Independence implies $\rho = 0$, never the reverse. The standard counterexample: $X$ symmetric
around 0, $Y = X^2$.

$$
\operatorname{Cov}(X, X^2) = E[X^3] - E[X]\,E[X^2] = 0 - 0 = 0
$$

$Y$ is a deterministic function of $X$, yet $\rho = 0$. Correlation only sees the linear part.

### Sampling noise of $\hat\rho$

For $n$ observations from a bivariate normal, approximately

$$
\operatorname{SE}(\hat\rho) \approx \frac{1-\rho^2}{\sqrt{n}}
$$

A better confidence interval uses Fisher's transformation $z = \operatorname{artanh}(\hat\rho)$,
which is close to normal with standard error $1/\sqrt{n-3}$:

$$
\rho \in \tanh\!\left(\operatorname{artanh}(\hat\rho) \pm \frac{1.96}{\sqrt{n-3}}\right) \quad (95\%)
$$

## 3. Worked Examples

### Example 1: higher correlation, lower beta

Market volatility $\sigma_M = 15\%$ (annual).

| Stock | $\sigma$ | $\rho$ with market | $\beta = \rho\,\sigma/\sigma_M$ |
|---|---|---|---|
| Tech stock | 30% | 0.5 | $0.5 \cdot 30/15 = 1.0$ |
| Utility | 10% | 0.9 | $0.9 \cdot 10/15 = 0.6$ |

The utility tracks the market more **reliably** (higher $\rho$), the tech stock moves **more**
with it (higher $\beta$).

### Example 2: same correlation, different destinations

Two funds with monthly returns $\rho = 0.95$ and identical 4% monthly volatility, but mean
returns of 0.8% vs 0.3% per month. The correlation says nothing about the 0.5% monthly gap:
after 10 years the expected log-wealth gap is roughly $120 \cdot 0.005 = 0.60$, i.e. one fund
ends with about $e^{0.6} \approx 1.8$ times the other's wealth.

### Example 3: one number for two regimes

Daily returns, zero means. Calm days (90%): both $\sigma = 1\%$, $\rho = 0.2$.
Crisis days (10%): both $\sigma = 3\%$, $\rho = 0.9$. The full-sample moments mix the regimes:

$$
\begin{aligned}
\operatorname{Cov} &= 0.9\,(0.2 \cdot 0.01^2) + 0.1\,(0.9 \cdot 0.03^2) = 0.18 \cdot 10^{-4} + 0.81 \cdot 10^{-4} = 0.99 \cdot 10^{-4} \\
\operatorname{Var} &= 0.9\,(0.01^2) + 0.1\,(0.03^2) = 1.8 \cdot 10^{-4} \\
\rho_{\text{full}} &= 0.99 / 1.8 = 0.55
\end{aligned}
$$

The quoted 0.55 is wrong in both regimes: too high for normal days, far too low for the days
diversification is supposed to protect you.

### Example 4: how wide is the interval?

$\hat\rho = 0.3$ from 36 monthly returns (3 years). Fisher:
$\operatorname{artanh}(0.3) = 0.310$, half-width $1.96/\sqrt{33} = 0.341$:

$$
\rho \in [\tanh(-0.031),\ \tanh(0.651)] = [-0.03,\ 0.57]
$$

Three years of data cannot distinguish "uncorrelated" from "moderately correlated".

## 4. Finance Application

### Beta is what you hedge with, correlation is how well it works

To neutralize market exposure you short $\beta$ units of the index, not $\rho$ units.
Correlation tells you what fraction of the variance the hedge removes ($\rho^2$, the $R^2$ of
the regression). Module 03/03 derives this as the minimum-variance hedge ratio.

### "Highly correlated" does not mean "same performance"

Two share classes of the same fund, one with a 1.5% fee and one with 0.5%, have return
correlation near 1 and a performance gap that compounds every year. Pairs trading relies on two
prices staying **together** (cointegration), which is a statement about levels and means that
return correlation cannot make.

### Never correlate price levels

Two independent assets that both trend upward have strongly correlated **prices**: both are
just correlated with time. Any series with a trend (equity indices, GDP, house prices, AuM)
shows this spurious correlation. Compute correlations on returns.

### Correlations rise in crises

Equity correlations measured over calm years understate the correlations seen in sell-offs,
exactly when diversification is needed. Risk managers look at rolling, stressed and tail
correlations, not a single full-sample $\rho$.

### Non-linear exposures

Options, convertible bonds, and strategies that sell volatility have payoffs that are curved in
the underlying. A delta-hedged long straddle has P&L roughly proportional to the **squared**
market move: near-zero correlation with the market, yet completely driven by it.

### The usual misreading

"Low correlation, so it diversifies." Check four things first: was it computed on returns,
over enough data, does it hold in the left tail, and is the dependence linear?

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
periods_per_year = 252

### Correlation vs beta (Example 1)

Simulate daily returns with the annual parameters from Example 1 and estimate both numbers.

In [ ]:
def correlated_normals(n, sigma_x, sigma_y, rho, rng):
    # Draw (X, Y) with given volatilities and correlation, zero mean
    z1 = rng.standard_normal(n)
    z2 = rng.standard_normal(n)
    x = sigma_x * z1
    y = sigma_y * (rho * z1 + np.sqrt(1 - rho**2) * z2)
    return x, y


def beta(x, y):
    return np.cov(x, y)[0, 1] / np.var(x, ddof=1)


n_days = 10 * periods_per_year
daily = lambda annual_vol: annual_vol / np.sqrt(periods_per_year)

rows = []
for name, vol, rho in [("tech", 0.30, 0.5), ("utility", 0.10, 0.9)]:
    market_r, stock_r = correlated_normals(n_days, daily(0.15), daily(vol), rho, rng)
    rows.append({
        "stock": name,
        "correlation": np.corrcoef(market_r, stock_r)[0, 1],
        "beta": beta(market_r, stock_r),
        "theory beta": rho * vol / 0.15,
    })

pd.DataFrame(rows).set_index("stock").round(3)

### Uncorrelated but fully dependent: a delta-hedged straddle

Daily market return `x` (decimal, 1% daily volatility). The straddle P&L is modelled as
proportional to `x**2` minus the option premium decay, so its expected value is 0.

In [ ]:
market_r = rng.normal(0, 0.01, size=100_000)
straddle_pnl = market_r**2 - 0.01**2   # gamma gain minus theta, in units of notional

print(f"corr(market, straddle P&L): {np.corrcoef(market_r, straddle_pnl)[0, 1]:+.3f}")
print(f"corr(|market|, straddle P&L): {np.corrcoef(np.abs(market_r), straddle_pnl)[0, 1]:+.3f}")

Correlation with the market is about zero. Correlation with the **size** of the market move is
close to one. The dependence is there, it is just not linear.

### Spurious correlation of price levels

Two **independent** assets, each with 7% annual drift and 20% annual volatility. Compare the
correlation of their price levels with the correlation of their returns, over many simulated
5-year histories.

In [ ]:
n_pairs = 500
n_days = 5 * periods_per_year
mu_daily = 0.07 / periods_per_year
vol_daily = daily(0.20)

level_corr, return_corr = [], []
for _ in range(n_pairs):
    log_r = rng.normal(mu_daily, vol_daily, size=(n_days, 2))   # independent columns
    prices = 100 * np.exp(np.cumsum(log_r, axis=0))             # CHF, start at 100
    level_corr.append(np.corrcoef(prices[:, 0], prices[:, 1])[0, 1])
    return_corr.append(np.corrcoef(log_r[:, 0], log_r[:, 1])[0, 1])

level_corr, return_corr = np.array(level_corr), np.array(return_corr)
print(f"price levels: mean corr {level_corr.mean():+.2f}, share with |corr| > 0.5: {np.mean(np.abs(level_corr) > 0.5):.0%}")
print(f"returns:      mean corr {return_corr.mean():+.3f}, share with |corr| > 0.5: {np.mean(np.abs(return_corr) > 0.5):.0%}")

### Same correlation, different destinations (Example 2)

Monthly returns, $\rho = 0.95$, equal 4% volatility, means 0.8% and 0.3%.

In [ ]:
n_months = 120
fund_a, fund_b = correlated_normals(n_months, 0.04, 0.04, 0.95, rng)
fund_a += 0.008
fund_b += 0.003

wealth_a = 1_000 * np.cumprod(1 + fund_a)   # CHF 1,000 invested
wealth_b = 1_000 * np.cumprod(1 + fund_b)

print(f"monthly return correlation: {np.corrcoef(fund_a, fund_b)[0, 1]:.3f}")
print(f"final wealth A: CHF {wealth_a[-1]:,.0f}   B: CHF {wealth_b[-1]:,.0f}   ratio {wealth_a[-1] / wealth_b[-1]:.2f}")

### Regime-dependent correlation (Example 3)

Ten years of daily returns. Three crisis blocks with high volatility and high correlation,
calm days otherwise.

In [ ]:
n_days = 10 * periods_per_year
crisis = np.zeros(n_days, dtype=bool)
for start, length in [(400, 80), (1300, 120), (2000, 60)]:
    crisis[start:start + length] = True

calm_x, calm_y = correlated_normals(n_days, 0.01, 0.01, 0.2, rng)
crisis_x, crisis_y = correlated_normals(n_days, 0.03, 0.03, 0.9, rng)
asset_x = np.where(crisis, crisis_x, calm_x)
asset_y = np.where(crisis, crisis_y, calm_y)

regimes = pd.DataFrame({"x": asset_x, "y": asset_y, "crisis": crisis})
print(f"share of crisis days:   {crisis.mean():.1%}")
print(f"full-sample correlation: {regimes['x'].corr(regimes['y']):.2f}")
print(regimes.groupby("crisis")[["x", "y"]].corr().xs("x", level=1)["y"].rename("correlation").round(2))

# Correlation on the worst 5% of days for asset x
worst = regimes["x"] <= regimes["x"].quantile(0.05)
print(f"correlation on x's worst 5% days: {regimes.loc[worst, 'x'].corr(regimes.loc[worst, 'y']):.2f}")

The crisis share here is about 10%, so the full-sample number lands near the 0.55 from
Example 3. Conditioning on bad days is itself tricky (selecting on $x$ shrinks its variance and
biases the correlation), which is one reason tail dependence gets its own tools.

### How noisy is a sample correlation? (Example 4)

In [ ]:
def fisher_ci(rho_hat, n, z_crit=1.96):
    z = np.arctanh(rho_hat)
    half_width = z_crit / np.sqrt(n - 3)
    return np.tanh(z - half_width), np.tanh(z + half_width)


lo, hi = fisher_ci(0.3, 36)
print(f"rho_hat = 0.30, n = 36 months: 95% CI [{lo:+.2f}, {hi:+.2f}]")

# Sampling distribution: true rho = 0.3, many 3-year monthly histories
true_rho, n_months, n_sims = 0.3, 36, 10_000
sample_rhos = np.empty(n_sims)
for i in range(n_sims):
    x, y = correlated_normals(n_months, 0.04, 0.04, true_rho, rng)
    sample_rhos[i] = np.corrcoef(x, y)[0, 1]

print(f"std of sample rho: {sample_rhos.std():.3f}   (approx. (1 - rho^2)/sqrt(n) = {(1 - true_rho**2) / np.sqrt(n_months):.3f})")
print(f"share of histories with sample rho < 0.1: {np.mean(sample_rhos < 0.1):.0%}")

### One crash day: Pearson vs rank correlation

One year of uncorrelated daily returns, plus a single day where both assets fall 15%.
Spearman's rank correlation replaces values by their ranks, so one extreme point carries
little weight.

In [ ]:
x, y = correlated_normals(periods_per_year, 0.01, 0.01, 0.0, rng)
returns = pd.DataFrame({"x": x, "y": y})
with_crash = pd.concat([returns, pd.DataFrame({"x": [-0.15], "y": [-0.15]})], ignore_index=True)

print(f"without crash: Pearson {returns['x'].corr(returns['y']):+.2f}   Spearman {returns['x'].corr(returns['y'], method='spearman'):+.2f}")
print(f"with crash:    Pearson {with_crash['x'].corr(with_crash['y']):+.2f}   Spearman {with_crash['x'].corr(with_crash['y'], method='spearman'):+.2f}")

Neither answer is "right": the crash day is real and a risk manager cares about it. The point is
that one day out of 253 can shift Pearson's $\rho$ by about 0.5, while Spearman barely moves.

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Straddle: zero correlation, perfect dependence
axes[0].scatter(market_r[:3000], straddle_pnl[:3000], s=4, alpha=0.4)
axes[0].set_title("Straddle P&L vs market: ρ ≈ 0, yet fully dependent")
axes[0].set_xlabel("daily market return")
axes[0].set_ylabel("straddle P&L (share of notional)")

# Spurious level correlation
bins = np.linspace(-1, 1, 41)
axes[1].hist(level_corr, bins=bins, alpha=0.7, label="price levels")
axes[1].hist(return_corr, bins=bins, alpha=0.7, label="daily returns")
axes[1].set_title("Independent assets: correlation of levels vs returns")
axes[1].set_xlabel("sample correlation")
axes[1].set_ylabel("number of simulated pairs")
axes[1].legend()

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(wealth_a, label=f"fund A (mean 0.8%/month)")
plt.plot(wealth_b, label=f"fund B (mean 0.3%/month)")
plt.title("Return correlation 0.95, very different outcomes")
plt.xlabel("month")
plt.ylabel("wealth (CHF)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
window = 63   # about 3 months of trading days
rolling_corr = regimes["x"].rolling(window).corr(regimes["y"])

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(rolling_corr, label=f"{window}-day rolling correlation")
ax.axhline(regimes["x"].corr(regimes["y"]), color="grey", linestyle="--", label="full-sample correlation")
ax.fill_between(regimes.index, -0.4, 1.0, where=regimes["crisis"], color="tab:red", alpha=0.15, label="crisis regime")
ax.set_ylim(-0.4, 1.0)
ax.set_title("One full-sample number hides two regimes")
ax.set_xlabel("trading day")
ax.set_ylabel("correlation")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(sample_rhos, bins=60, density=True, alpha=0.75)
plt.axvline(true_rho, color="black", linestyle="--", label=f"true ρ = {true_rho}")
plt.axvline(0, color="tab:red", linestyle=":", label="ρ = 0")
plt.title(f"Sample correlation from {n_months} monthly returns, true ρ = {true_rho}")
plt.xlabel("sample correlation")
plt.ylabel("density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 7. Exercises

### Basic

1. A stock has 25% annual volatility and correlation 0.6 with an index of 18% volatility.
   What is its beta? What fraction of its variance does a beta-hedge with the index remove?
2. Using `fisher_ci`, compute the 95% interval for $\hat\rho = 0.5$ from 60 monthly returns,
   and from 10 years of daily returns.

### Intermediate

3. Redo Example 3 with calm days 80% ($\sigma = 1\%$, $\rho = 0$) and crisis days 20%
   ($\sigma = 2.5\%$, $\rho = 0.8$). Compute the full-sample correlation by hand, then check it
   by simulation.
4. Interpret: a brochure says "gold has a correlation of 0.05 with Swiss equities over the last
   three years, so it protects your portfolio in a crash". List what this number does and does
   not support.

### Advanced

5. Show that $\operatorname{Cov}(X, X^2) = 0$ for any $X$ symmetric around 0 with finite third
   moment. Then let $X$ be negatively skewed (equity-like). What sign does
   $\operatorname{Cov}(X, X^2)$ take, and what does that mean for the market correlation of a
   long-volatility position?

## 8. Solutions / Checks

In [ ]:
# 1. beta = rho * sigma / sigma_index, variance removed = rho^2
rho, vol, index_vol = 0.6, 0.25, 0.18
print(f"1. beta = {rho * vol / index_vol:.3f},  variance removed = {rho**2:.0%}")

# 2.
for n in [60, 10 * periods_per_year]:
    lo, hi = fisher_ci(0.5, n)
    print(f"2. n = {n:5d}: 95% CI [{lo:.2f}, {hi:.2f}]")

# 3. By hand, then by simulation
p_calm, p_crisis = 0.8, 0.2
cov_mix = p_calm * 0.0 * 0.01**2 + p_crisis * 0.8 * 0.025**2
var_mix = p_calm * 0.01**2 + p_crisis * 0.025**2
print(f"3. full-sample rho by hand: {cov_mix / var_mix:.3f}")

n = 1_000_000
is_crisis = rng.random(n) < p_crisis
cx, cy = correlated_normals(n, 0.01, 0.01, 0.0, rng)
kx, ky = correlated_normals(n, 0.025, 0.025, 0.8, rng)
print(f"   by simulation:           {np.corrcoef(np.where(is_crisis, kx, cx), np.where(is_crisis, ky, cy))[0, 1]:.3f}")

**4.** Supported: over those three years, gold's returns had almost no **linear** co-movement
with Swiss equities on average. Not supported:
- **Crash protection** is a statement about the left tail; a full-sample $\rho$ averages calm
  and crisis days (Example 3). Look at returns on the worst equity days instead.
- **Precision**: with 36 monthly returns the 95% interval around 0.05 runs from about $-0.28$
  to $+0.37$ (Example 4).
- **Size of the offset**: even with negative correlation, protection depends on beta and
  position size, not $\rho$ alone.
- **Units**: gold is priced in USD, so a CHF investor's gold return includes the currency.

**5.** $E[X^3] = 0$ by symmetry and $E[X] = 0$, so $\operatorname{Cov}(X, X^2) = E[X^3] - E[X]E[X^2] = 0$.
For negatively skewed $X$ (zero mean), $E[X^3] < 0$, so $\operatorname{Cov}(X, X^2) < 0$: large
squared moves come mostly with down moves. A long-volatility position (P&L $\propto X^2$) then
has **negative** correlation with the market. That is the empirical reason volatility is
treated as an equity hedge, and it comes from skewness, not from any linear link.

## 9. Conclusion

- $\rho$ is the covariance of standardized returns: it discards means and volatilities by design.
- Size of co-movement is **beta** $= \rho\,\sigma_Y/\sigma_X$; quality of a hedge is $\rho^2$.
- Correlated returns can still lead to very different wealth; correlate **returns**, never price levels.
- $\rho = 0$ does not mean independent: curved payoffs (options, volatility strategies) hide from it.
- One full-sample $\rho$ averages regimes, and crisis correlation is the one that matters for diversification.
- Sample correlations from a few years of data are noisy: report a Fisher interval, not just $\hat\rho$.

Next: [03 Minimum-variance hedge ratio](./03_minimum_variance_hedge_ratio.ipynb).